In [11]:
print("RESULTS COLUMNS:\n", results.columns)
print("\nRACES COLUMNS:\n", races.columns)
print("\nCONSTRUCTORS COLUMNS:\n", constructors.columns)

RESULTS COLUMNS:
 Index(['resultId', 'raceId', 'driverId', 'constructorId', 'number', 'grid',
       'position', 'positionText', 'positionOrder', 'points', 'laps', 'time',
       'milliseconds', 'fastestLap', 'rank', 'fastestLapTime',
       'fastestLapSpeed', 'statusId', 'constructor'],
      dtype='object')

RACES COLUMNS:
 Index(['raceId', 'year', 'round', 'circuitId', 'name', 'date', 'time', 'url',
       'fp1_date', 'fp1_time', 'fp2_date', 'fp2_time', 'fp3_date', 'fp3_time',
       'quali_date', 'quali_time', 'sprint_date', 'sprint_time'],
      dtype='object')

CONSTRUCTORS COLUMNS:
 Index(['constructorId', 'constructorRef', 'name', 'nationality', 'url'], dtype='object')


In [12]:
season_races = races[races["year"] == SEASON]
season_race_ids = season_races["raceId"]

results = results[results["raceId"].isin(season_race_ids)]

In [13]:
results = results.merge(
    constructors[["constructorId", "name"]],
    on="constructorId",
    how="left"
)

results = results.rename(columns={"name": "constructor"})

In [14]:
results["position"] = pd.to_numeric(results["position"], errors="coerce")

In [15]:
clean_results = results[results["position"].notna()].copy()

In [16]:
print(clean_results.head())
print("\nUnique constructors:\n", clean_results["constructor"].unique())
print("\nPositions range:\n", clean_results["position"].min(), "to", clean_results["position"].max())

   resultId  raceId  driverId  constructorId number  grid  position  \
0     26765    1145       846              1      4     1         1   
1     26766    1145       830              9      1     3         2   
2     26767    1145       847            131     63     4         3   
3     26768    1145       863            131     12    16         4   
4     26769    1145       848              3     23     6         5   

  positionText  positionOrder  points  laps         time milliseconds  \
0            1              1    25.0    57  1:42:06.304      6126304   
1            2              2    18.0    57       +0.895      6127199   
2            3              3    15.0    57       +8.481      6134785   
3            4              4    12.0    57      +10.135      6136439   
4            5              5    10.0    57      +12.773      6139077   

  fastestLap rank fastestLapTime fastestLapSpeed  statusId constructor  \
0         43    1       1:22.167              \N         1  

AttributeError: 'DataFrame' object has no attribute 'unique'

In [19]:
PU_MAPPING = {
    "Red Bull": "Honda",
    "RB F1 Team": "Honda",

    "Mercedes": "Mercedes",
    "McLaren": "Mercedes",
    "Aston Martin": "Mercedes",
    "Williams": "Mercedes",

    "Ferrari": "Ferrari",
    "Haas F1 Team": "Ferrari",
    "Sauber": "Ferrari",

    "Alpine F1 Team": "Renault"
}

In [23]:
results = results.merge(
    constructors[["constructorId", "name"]],
    on="constructorId",
    how="left"
)

# Remove any old constructor columns if they exist
results = results.drop(columns=[col for col in results.columns if "constructor" in col and col != "constructorId" and col != "name"], errors="ignore")

# Rename cleanly
results = results.rename(columns={"name": "constructor"})

In [27]:
print(type(clean_results["constructor"]))

clean_results["power_unit"] = clean_results["constructor"].map(PU_MAPPING)
print(clean_results[["constructor", "power_unit"]].drop_duplicates())

<class 'pandas.core.frame.DataFrame'>


AttributeError: 'DataFrame' object has no attribute 'map'

In [28]:
print([col for col in clean_results.columns if "constructor" in col])

['constructorId', 'constructor', 'constructor']


In [33]:
clean_results = clean_results.loc[:, ~clean_results.columns.duplicated()]

# If you see constructor_x / constructor_y, pick the correct one
if "constructor_x" in clean_results.columns:
    clean_results["constructor"] = clean_results["constructor_x"]

elif "constructor_y" in clean_results.columns:
    clean_results["constructor"] = clean_results["constructor_y"]
    
clean_results = clean_results.drop(
    columns=[col for col in clean_results.columns if col.startswith("constructor_")],
    errors="ignore"
)

print(type(clean_results["constructor"]))

<class 'pandas.core.series.Series'>


In [34]:
clean_results["power_unit"] = clean_results["constructor"].map(PU_MAPPING)

In [35]:
print(clean_results[clean_results["power_unit"].isna()]["constructor"].unique())

[]


In [36]:
race_groups = {}

for raceId, grp in clean_results.groupby("raceId"):
    race_df = grp.sort_values("position")[["position", "power_unit"]].copy()
    race_groups[raceId] = race_df.reset_index(drop=True)

In [37]:
sample_race = list(race_groups.keys())[0]
print(race_groups[sample_race].head(10))

   position power_unit
0         1   Mercedes
1         2      Honda
2         3   Mercedes
3         4   Mercedes
4         5   Mercedes
5         6   Mercedes
6         7    Ferrari
7         8    Ferrari
8         9   Mercedes
9        10    Ferrari


In [38]:
def compute_exponential_scores(race_groups, k=3):
    pu_scores = {}

    for raceId, race_df in race_groups.items():
        max_pos = race_df["position"].max()

        # Avoid division issues
        if max_pos == 1:
            race_df["p_norm"] = 0
        else:
            race_df["p_norm"] = (race_df["position"] - 1) / (max_pos - 1)

        # Exponential decay scoring
        race_df["score"] = np.exp(-k * race_df["p_norm"])

        # Average per PU in that race
        pu_race_scores = race_df.groupby("power_unit")["score"].mean()

        for pu, s in pu_race_scores.items():
            pu_scores.setdefault(pu, []).append(s)

    # Final aggregation
    final_scores = {pu: np.mean(scores) for pu, scores in pu_scores.items()}

    return final_scores

In [39]:
scores = compute_exponential_scores(race_groups, k=3)

exp_df = pd.DataFrame({
    "power_unit": list(scores.keys()),
    "score": list(scores.values())
}).sort_values("score", ascending=False).reset_index(drop=True)

exp_df["rank"] = exp_df.index + 1

print(exp_df)

  power_unit     score  rank
0   Mercedes  0.418294     1
1      Honda  0.329242     2
2    Ferrari  0.274593     3
3    Renault  0.124997     4


In [41]:
comparison = pu_table[["power_unit", "rank"]].rename(columns={"rank": "colley_rank"})

comparison = comparison.merge(
    exp_df[["power_unit", "rank"]].rename(columns={"rank": "exp_rank"}),
    on="power_unit"
)

plt.figure(figsize=(8,5))

plt.plot(comparison["power_unit"], comparison["colley_rank"], marker='o', label="Colley")
plt.plot(comparison["power_unit"], comparison["exp_rank"], marker='o', label="Exponential")

plt.gca().invert_yaxis()

plt.xlabel("Power Unit")
plt.ylabel("Rank (Lower = Better)")
plt.title("Colley vs Exponential Ranking")

plt.legend()
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.show()

NameError: name 'pu_table' is not defined